In [ ]:
# 1. Install PyCaret
# We use --ignore-installed blinker to prevent errors with the pre-installed version
!pip install pycaret --ignore-installed blinker

# 2. Patch PyCaret to support Python 3.12
# PyCaret currently has a check that raises an error on Python 3.12+.
# We will modify the installed file to bypass this check.
import os
import sys

# Define the path to the file (Standard Colab path)
# We iterate through sys.path to be robust
pycaret_init_path = None
for path in sys.path:
    potential_path = os.path.join(path, 'pycaret', '__init__.py')
    if os.path.exists(potential_path):
        pycaret_init_path = potential_path
        break

if pycaret_init_path:
    print(f"Found PyCaret at: {pycaret_init_path}")
    with open(pycaret_init_path, 'r') as f:
        content = f.read()

    # Replace the blocking check with a harmless condition
    target_str = "sys.version_info >= (3, 12)"
    if target_str in content:
        print("Patching version check...")
        new_content = content.replace(target_str, "False")
        with open(pycaret_init_path, 'w') as f:
            f.write(new_content)
        print("PyCaret patched successfully!")
    else:
        print("Version check not found (already patched?).")
else:
    print("Could not find PyCaret installation to patch.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 6.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 14.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 1

Found PyCaret at: /usr/local/lib/python3.12/dist-packages/pycaret/__init__.py
Patching version check...
PyCaret patched successfully!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from pycaret.clustering import *

# 1. Generate "Moons" Data (Non-spherical data)
# We use sklearn to make the data, then convert to Pandas for PyCaret
X, _ = make_moons(n_samples=500, noise=0.05, random_state=42)

# Convert to DataFrame (PyCaret requires a Pandas DataFrame)
data = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])

# Visualize the raw data
plt.figure(figsize=(8, 5))
plt.scatter(data['Feature_1'], data['Feature_2'], s=30, color='gray', alpha=0.6)
plt.title("Input Data: Two Moons (Difficult for K-Means)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

# Display first few rows
print(data.head())

In [ ]:
# 2. Initialize PyCaret Setup
# 'session_id' ensures reproducibility
# 'verbose=False' prevents it from printing a massive configuration table
experiment = setup(data, session_id=42, verbose=False)

print("PyCaret Environment Setup Complete.")

In [ ]:
# 3. Create the DBSCAN Model
# id='dbscan' triggers the DBSCAN algorithm
# eps=0.2 is the radius (epsilon) that defines the neighborhood
dbscan_model = create_model('dbscan', eps=0.2, min_samples=5)

# 4. Assign Labels to the Data
# This adds a 'Cluster' column to our original DataFrame
results = assign_model(dbscan_model)

# Show the results with the new Cluster labels
print(results.head())

# Note: Cluster 'Cluster -1' represents outliers/noise in DBSCAN

In [ ]:
# 5. Visualize the Clusters using PyCaret
# We use the 'cluster' plot type to see the 2D representation
plot_model(dbscan_model, plot='cluster')

In [ ]:
# Custom Plot to verify the "Moons" shape detection
plt.figure(figsize=(10, 6))

# Define colors (Outliers usually black or gray)
# PyCaret saves labels as strings like 'Cluster 0', 'Cluster 1', 'Cluster -1'
colors = {'Cluster 0': 'red', 'Cluster 1': 'blue', 'Cluster -1': 'black'}

for cluster in results['Cluster'].unique():
    subset = results[results['Cluster'] == cluster]
    plt.scatter(subset['Feature_1'], subset['Feature_2'],
                label=cluster, s=50, alpha=0.7)

plt.title("DBSCAN Result: Perfect Separation of Moons")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()